### 7.1 – Analyse critique 

#### A) Quelles sont les limites de l’approche bag-of-words / TF-IDF pour l’analyse de sentiment ? Donnez au moins 3 exemples concrets de tweets que votre modèle classerait probablement mal, et expliquez pourquoi.

D'après ce qu'on a appris, l'approche TF IDF traite les mots individuellement sans tenir compte de leur ordre ou de leur contexte grammatical donc cela induit une perte de contexte comme on l'a vu précédemment dans les notebooks, si une phrase contient de la négation, donc les mots précédés de la négation sont considérés dans leur sens premier. On peut aussi faire face à de l'ironie ou du sarcasmen qui fausse aussi les résultats. Finalement, comme en français, l'anglais a des mots qui ont plusieurs sens, notamment dans l'argot des jeunes sur les réseaux sociaux, donc un mot peut être identifié comme négatif alors que c'est une expression nouvelle.

Pour citer 3 exemples concrets de tweets que le modèle classerait probablement mal, je pense à :
- pour la négation : "Ce film n'est pas mauvais" -> qui serait classé en positif car la négation n'est pas vraiment prise en compte
- pour l'ironie : "Super, mon train a du retard" -> serait classé en positif aussi car "super" semble positif
- pour l'argot : "Ce plat est une tuerie" -> le mot "tuerie" est sémantiquement négatif mais ici il est utilisé pour dire que le plat est délicieux


#### B) Obsolescence du dataset. Le dataset Sentiment140 date de 2009. Quels problèmes cela pose-t-il pour une utilisation en 2026 ? Proposez des solutions concrètes.

Le fait que le dataset date de 2009 est problématique pour l'analyse car le langage a évolué depuis, donc les néologismes et l'argot moderne ne sont pas reconnus par la modèle. On peut aussi penser aux émojis qui peuvent rajouter du contexte différent qu'à l'époque, mais comme on les a enlevé du modèle on en tient pas compte.

En termes de solutions concrètes, j'ai cherché et trouvé que le fine-tuning pouvait aider, car cela consiste en un ré entrainement régulier du modèle avec des petits échantillons de tweets récents labellisés à la main.

J'ai également trouvé qu'on pouvait utiliser des modèles pré-entraînés modernes (comme BERT), qui comprennent le contexte sémantique et les émojis, plutôt que de se baser uniquement sur la fréquence des mots.

### 7.2 – Conception d’un système d’alerte intelligent (voir la fonctionnalité dans le dashboard : def get_smart_alert_message)

Une marque vous demande de détecter les « crises de réputation » en temps réel. Concevez un
système d’alerte qui :

— Définit des critères de déclenchement pertinents (pas un simple seuil fixe).

— Minimise les faux positifs.

— S’adapte au volume normal de la marque.

#### Approche employée : le z-score

Plutôt que de mettre un seuil fixe, qui déclencherait des fausses alertes aux heures de pointe et raterait des crises la nuit, j'ai implémenté un Z-Score dynamique. (définition : "Le Z-score, également connu sous le nom de score standard, est une mesure statistique qui décrit la position d’une valeur donnée par rapport à la moyenne d’un groupe de valeurs. ")

Le système calcule la moyenne et l'écart-type des minutes précédentes, et l'alerte se déclenche uniquement si le volume actuel dépasse Moyenne + 3 * Écart-Type. Une hausse soudaine et brutale sera détectée, peu importe l'heure de la journée.

### 7.3 – Indicateur original

Proposez et implémentez un indicateur streaming qui n’est pas demandé dans les exercices.

#### Indicateur : Vitesse de propagation des Hashtags (Viralité)

J'ai choisi de ne pas seulement regarder le sentiment, mais aussi les sujets émergents (" les Trending Topics").

En effet, le sentiment nous dit comment les gens se sentent, mais pas pourquoi. 
Alors, en extrayant les hashtags les plus fréquents sur une fenêtre courte (les 1000 derniers tweets), on peut détecter le début d'un bad buzz (un boycott par exemple) ou une tendance virale avant même qu'elle ne soit majoritaire dans le volume global.

### 7.4 Réflexion architecturale

#### A) TCP vs Kafka. Comparez le socket TCP (Phase 1) et Kafka (Phase 2) : avantages, inconvénients, cas d’usage.

Dans la Phase 1, on utilisait un Socket TCP qui est comme un tuyau simple car ça marche bien pour tester (c'est facile à lancer avec nc -lk), mais c'est synchrone et fragile. Donc si Spark plante ou si on coupe le notebook, les données envoyées pendant ce temps sont perdues à jamais.

Dans la Phase 2, on est passé à Kafka qui est un système asynchrone avec un buffer. Kafka stocke les messages sur son disque donc si Spark plante, quand je le relance, il peut lire ce qu'il a raté grâce aux offsets. Donc finalement, c'est beaucoup plus robuste pour la production.

#### B) Back-pressure. Que se passe-t-il si le consumer Spark est plus lent que le producer ? Comment Kafka gère-t-il cette situation par rapport à TCP ?

S le Producer envoie 1000 tweets par seconde mais que le modèle Spark ne peut en traiter que 500, le résultat est différent selon l'approche utilisée :
- Avec TCP, le tuyau sature (buffer overflow) et les données excédentaires sont rejetées et perdues.
- Avec Kafka, il garde les messages en attente sur son disque (Retention). Alors Spark accumule du retard (Lag), mais aucune donnée n'est perdue. Spark traitera le stock plus tard quand le pic de charge sera passé.

#### C) Passage à l’échelle. Comment passeriez-vous à l’échelle pour traiter 1 million de tweets par seconde ? Identifiez les goulots d’étranglement et proposez des solutions.

Mon architecture actuelle ne tiendrait pas la charge du million de tweets par seconde. 

Pour traiter 1 million de tweets/sec, il faudrait paralléliser massivement !

Du côté de kafka, il faudrait augmenter le nombre de partitions du Topic (passer de 1 à 100 ou 1000 partitions). C'est ce qui permet de lire en parallèle.

Du côté de Spark, on devrait ajouter des machines (Workers) au cluster et augmenter le nombre d'exécuteurs. Dans tous les cas, il faut que chaque partition Kafka soit lue par un exécuteur Spark.

Les goulots d'étranglement identifiés sont :
- l'inférence du modèle (car calculer la prédiction prend du CPU)
- l'écriture sur HDFS (HDFS ne supporte pas l'écriture de millions de petits fichiers JSON).

Donc il faudrait probablement écrire les résultats dans une base de données rapide (NoSQL) plutôt que des fichiers à plat.